In [ ]:
import os
import matplotlib.pyplot as plt
import torch
import torchvision
import numpy as np
from torch import nn
import torchvision.transforms as T
from torch.nn import functional as F

def one_hot(x, length=10):
    out = torch.zeros(length)
    out[x] = 1.
    return out

target_transform = T.Compose([
    T.Lambda(one_hot)
])

transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5,),(0.5,))
    ])

train_data = torchvision.datasets.MNIST(root="data/mnist", train=True, download=True,
                                        transform=transform, target_transform=target_transform)

train_batch_size = 128
num_workers = 2
train_dataloader = torch.utils.data.DataLoader(dataset=train_data, batch_size=train_batch_size, num_workers=num_workers, shuffle=True)

class GeneratorBlock(nn.Module):
  def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int, padding: int, act: nn.Module = nn.ReLU()):
    super(GeneratorBlock, self).__init__()
    self.conv2d_trans = nn.ConvTranspose2d(
        in_channels=in_channels, out_channels=out_channels,
        kernel_size=kernel_size, padding=padding, stride=stride
    )
    self.batch_norm = nn.BatchNorm2d(out_channels)
    self.act = act

  def forward(self, x):
      return self.act(self.batch_norm(self.conv2d_trans(x)))
  
class Generator(nn.Module):
  def __init__(self, init_channels: int = 1024, latent_dim: int = 62, img_channels: int = 1, num_classes: int = 10, num_codes: int = 2):
    super(Generator, self).__init__()
    self.num_classes = num_classes
    self.num_codes = num_codes
    self.fc1 = nn.Sequential(
      nn.Linear(in_features=latent_dim+num_classes+num_codes, out_features=init_channels),
      nn.ReLU(), nn.BatchNorm1d(init_channels))
    self.fc2 = nn.Sequential(
      nn.Linear(in_features=init_channels, out_features=128*7*7),
      nn.ReLU(), nn.BatchNorm1d(128*7*7))
    self.conv = nn.Sequential(
        GeneratorBlock(in_channels=128, out_channels=64, stride=2, kernel_size=4, padding=1),
        # output shape: ???
        nn.ConvTranspose2d(in_channels=64, out_channels=img_channels, stride=2, kernel_size=4, padding=1),
        nn.Tanh()
    )

  def forward(self, z, c1, c2):
    out = torch.cat((z, c1, c2), dim=1)
    out = self.fc1(out)
    out = self.fc2(out)
    out = self.conv(out.view(z.size(0), 128, 7, 7))
    return out
  
class DiscriminatorBlock(nn.Module):
  def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int, padding: int, alpha: float = 0.1):
    super(DiscriminatorBlock, self).__init__()
    self.conv2d = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
    self.batch_norm = nn.BatchNorm2d(out_channels)
    self.leaky_relu = nn.LeakyReLU(alpha)
  
  def forward(self, x):
    return self.leaky_relu(self.batch_norm(self.conv2d(x)))
  
class Discriminator(nn.Module):
    def __init__(self, init_channels: int = 64, img_channels: int = 1, num_classes:int = 10, num_codes: int = 2, alpha: float = 0.1):
        super(Discriminator, self).__init__()
        self.num_classes = num_classes
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels=img_channels, out_channels=init_channels, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(alpha),
            # output shape: (init_channels, 14, 14)
            DiscriminatorBlock(in_channels=init_channels, out_channels=init_channels*2, kernel_size=4, stride=2, padding=1, alpha=alpha),
            # output shape: (init_channels*2, 7, 7)
        )
        self.fc_out = nn.Sequential(
            nn.Linear(in_features=init_channels*2*7*7, out_features=init_channels*16),
            nn.LeakyReLU(alpha),
            nn.BatchNorm1d(init_channels*16),
            nn.Linear(in_features=init_channels*16, out_features=1)
        )
        self.q_net = nn.Sequential(
            nn.Linear(in_features=init_channels*2*7*7, out_features=128), nn.BatchNorm1d(128), 
            nn.LeakyReLU(alpha),
            nn.Linear(in_features=128, out_features=num_classes+num_codes), 
        )
        
    def forward(self, x):
        out = self.conv(x)
        features = out.view(x.size(0), -1)
        out = self.fc_out(features) # real or fake
        q = self.q_net(features)
        c1 = q[:, :self.num_classes]
        c2 = q[:, self.num_classes:]
        return out, c1, c2

def update_discriminator(x, z, c1, c2, D, G, criterion, trainer_D):
  batch_size = x.shape[0]
  ones = torch.ones((batch_size,), device=x.device)
  zeros = torch.zeros((batch_size,), device=x.device)

  trainer_D.zero_grad()

  real_y, _, _ = D(x)
  fake_x = G(z,c1, c2)
  # Do not need to compute gradient for G, detach it from computing gradients.
  fake_y, _, _ = D(fake_x.detach())
  loss_D = (criterion(real_y, ones.reshape(real_y.shape)) +
            criterion(fake_y, zeros.reshape(fake_y.shape))) / 2

  loss_D.backward()
  trainer_D.step()
  return loss_D.item()

def update_generator(z, c1, c2, D, G, criterion, trainer_G, lambd=1.):
  batch_size = z.shape[0]
  ones = torch.ones((batch_size,), device=z.device)

  trainer_G.zero_grad()

  fake_x = G(z, c1, c2)
  fake_y, fake_c1, fake_c2 = D(fake_x)

  loss_adv = criterion(fake_y, ones.reshape(fake_y.shape))
  loss_cat = nn.CrossEntropyLoss(reduction="sum")(fake_c1, torch.argmax(c1, dim=1))
  loss_cont = lambd*nn.MSELoss(reduction="sum")(fake_c2, c2)
  loss_G = loss_adv + loss_cat + loss_cont
  loss_G.backward()
  trainer_G.step()
  return loss_G.item()


def init_weights(w):
    if isinstance(w, (nn.Linear, nn.Conv2d, nn.ConvTranspose2d)):
        torch.nn.init.normal_(w.weight.data, 0., 0.02)
        if w.bias is not None:
            torch.nn.init.constant_(w.bias.data, 0.0)
    elif isinstance(w, (nn.BatchNorm2d, nn.BatchNorm1d)):
        torch.nn.init.normal_(w.weight.data, 1.0, 0.02)
        torch.nn.init.constant_(w.bias.data, 0.0)


def train_dcgan(D, G, lr_D, lr_G, latent_dim, dataloader, num_codes, num_epochs, device,
                fixed_noise, fixed_label, fixed_code, lambd=1., visualize=True, print_every=50):
  print(f"Device: {device}")
  criterion = nn.BCEWithLogitsLoss(reduction="sum")

  D.apply(init_weights)
  G.apply(init_weights)

  trainer_D = torch.optim.Adam(D.parameters(), lr=lr_D, betas=(0.5, 0.999))
  trainer_G = torch.optim.Adam(G.parameters(), lr=lr_G, betas=(0.5, 0.999))

  # trainer_D = torch.optim.Adam(
  #     list(D.conv.parameters()) + list(D.fc_out.parameters()), # only D
  #     lr=lr_D, betas=(0.5,0.999))
  # trainer_G = torch.optim.Adam(
  #     list(G.parameters()) + list(D.q_net.parameters()), # G and q_net
  #     lr=lr_G, betas=(0.5,0.999))


  metrics = []

  fixed_noise = fixed_noise.to(device)
  fixed_label = fixed_label.to(device)
  fixed_code = fixed_code.to(device)

  for epoch in range(num_epochs):
    G.train()
    D.train()
    loss_D = 0.
    loss_G = 0.
    num_instances = 0
    for step_num, batch in enumerate(dataloader):
      x,c1 = batch
      x = x.to(device)
      c1 = c1.to(device)
      batch_size = x.shape[0]
      num_instances += batch_size
      z = torch.normal(0., 1., size=(batch_size, latent_dim), device=x.device)
      c2 = 2. * torch.rand(batch_size, num_codes, device=x.device) - 1.

      loss_D += update_discriminator(x=x, z=z, c1=c1, c2=c2, D=D, G=G, criterion=criterion, trainer_D=trainer_D)
      loss_G += update_generator(z=z, c1=c1, c2=c2, D=D, G=G, criterion=criterion, trainer_G=trainer_G, lambd=lambd)
      # scheduler_D.step()
      # scheduler_G.step()
      # if step_num % print_every==0:
      #   print(f"[Epoch {epoch}/{num_epochs}] [Step {step_num}/{len(dataloader)}] loss_D: {loss_D/num_instances:.4f}, loss_G: {loss_G/num_instances:.4f}")


    loss_D /= num_instances
    loss_G /= num_instances
    metrics.append([loss_D, loss_G])
    print(f"[Epoch {epoch}/{num_epochs}] loss_D: {loss_D:.4f}, loss_G: {loss_G:.4f}")

    G.eval()
    D.eval()
    
    if visualize:
      print("Generated images:")
      os.makedirs("visualizations", exist_ok=True)
      # z = torch.normal(0., 1., size=(16, latent_dim, 1, 1), device=x.device)
      fake_data = G(fixed_noise, fixed_label, fixed_code).detach().cpu()
      show_batch((fake_data,None), save_name=f"visualizations/generated-{epoch:02d}.png")
  metrics = np.array(metrics)
  plt.plot(metrics[:, 0], label="loss_D")
  plt.plot(metrics[:, 1], label="loss_G")
  plt.legend()
  plt.show()


lr_D = 2.e-4
lr_G = 2e-4
lambd = 0.1


num_epochs = 50
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

latent_dim = 62
num_codes = 2
num_classes = 10
discriminator = Discriminator(init_channels=64, num_classes=num_classes, num_codes=num_codes).to(device)
generator = Generator(init_channels=1024, latent_dim=latent_dim, num_classes=num_classes, num_codes=num_codes).to(device)
    
fixed_noise = torch.normal(0., 1., size=(10, latent_dim))
fixed_label = torch.cat([one_hot(i).view(1,-1) for i in range(10)], dim=0)
fixed_code = 2. * torch.rand(10, num_codes) - 1.
train_dcgan(
    D=discriminator, G=generator, lr_D=lr_D, lr_G=lr_G, lambd=lambd,
    latent_dim=latent_dim, num_codes=num_codes,dataloader=train_dataloader, num_epochs=num_epochs, 
    device=device, fixed_noise=fixed_noise, fixed_label=fixed_label, fixed_code=fixed_code, 
    visualize=True, print_every=100)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, init_channels: int = 64, num_classes: int = 10, num_codes: int = 2):
        super(Discriminator, self).__init__()
        self.num_classes = num_classes
        
        self.conv = nn.Sequential(
            # Layer 1: No BatchNorm for the very first layer
            nn.Conv2d(1, init_channels, 4, 2, 1),
            nn.LeakyReLU(0.1),
            # Layer 2: BatchNorm is fine here
            DiscriminatorBlock(init_channels, init_channels * 2, 4, 2, 1) # 7x7
        )
        
        # Flattened size is (init_channels * 2) * 7 * 7
        flat_size = init_channels * 2 * 7 * 7
        
        self.fc_out = nn.Linear(flat_size, 1)
        self.q_net = nn.Sequential(
            nn.Linear(flat_size, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1),
            nn.Linear(128, num_classes + num_codes)
        )
        
    def forward(self, x):
        features = self.conv(x)
        features = features.view(x.size(0), -1)
        
        validity = self.fc_out(features)
        q = self.q_net(features)
        
        c1 = q[:, :self.num_classes]
        c2 = q[:, self.num_classes:]
        return validity, c1, c2